# Question 1

In [0]:
from pyspark.sql.types import StructType, StructField, DateType, StringType
from datetime import date

data = [(date(2020, 6, 1), "Booked"), (date(2020, 6, 2), "Booked"), (date(2020, 6, 3), "Booked"), (date(2020, 6, 4), "Available"), (date(2020, 6, 5), "Available"), (date(2020, 6, 6), "Available"), (date(2020, 6, 7), "Booked")]

schema = StructType([StructField("show_date", DateType(), True), StructField("show_status", StringType(), True)])

df = spark.createDataFrame(data, schema)
display(df)

In [0]:
df.createOrReplaceTempView('shows')

In [0]:
%sql
WITH base AS (
    SELECT show_date, show_status, ROW_NUMBER() OVER (PARTITION BY show_status ORDER BY show_date) AS rn FROM shows
),
grp_data AS (
    SELECT show_date, show_status, date_sub(show_date, rn) AS grp FROM base
)
SELECT show_status, MIN(show_date) AS start_date, MAX(show_date) AS end_date FROM grp_data
GROUP BY show_status, grp ORDER BY start_date

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w = Window.partitionBy('show_status').orderBy('show_date')
df = df.withColumn('rn', F.row_number().over(w))\
       .withColumn('grp', F.expr("date_sub(show_date, rn)"))

result = df.groupBy('show_status','grp')\
           .agg(F.min('show_date').alias('start_date'), F.max('show_date').alias('end_date')).orderBy('start_date')
display(result)

# Question 2

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

data = [(1, "Alice", None, 10000), (2, "Bob", 1, 9000), (3, "Carol", 1, 12000),(4, "David", 2, 9500), (5, "Eve", 2, 8500)]
schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("emp_name", StringType(), True),
    StructField("manager_id", IntegerType(), True),
    StructField("salary", IntegerType(), True)
])

df = spark.createDataFrame(data, schema)
display(df)

In [0]:
df.createOrReplaceTempView('employees')

In [0]:
%sql
SELECT e.emp_id AS employee_id, e.emp_name AS employee_name, e.salary AS employee_salary, m.emp_name AS manager_name, m.salary AS manager_salary
FROM employees e JOIN employees m ON e.manager_id = m.emp_id WHERE e.salary > m.salary;

In [0]:
from pyspark.sql import functions as F

df = df.alias("e").join(df.alias("m"), F.col("e.manager_id") == F.col("m.emp_id"), "inner")\
    .filter(F.col("e.salary") > F.col("m.salary"))
    
result = df.select(F.col("e.emp_id").alias("employee_id"),
                   F.col("e.emp_name").alias("employee_name"),
                   F.col("e.salary").alias("employee_salary"),
                   F.col("m.emp_name").alias("manager_name"),
                   F.col("m.salary").alias("manager_salary")
                 )

display(result)

# Question 3

In [0]:
%sql
SELECT DATE_FORMAT(sale_date, 'yyyy-MM') AS month, product_id, AVG(quantity) AS avg_sales
FROM your_table_name
GROUP BY DATE_FORMAT(sale_date, 'yyyy-MM'), product_id
ORDER BY avg_sales DESC;

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
df1 = df.withColumn('rank', rank().over(Window.orderby('id')))
      df.withColumn('NewName', coalesce(when(col('rank') % 2 == 1, lead('name').over(Window.orderby('id')))
                                        .otherwise(lag('name').over(Window.orderby('id'))), col('name'))